In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# 1. ĐỌC DỮ LIỆU
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

# Chia tập Train/Test có phân tầng (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. TỰ ĐỘNG PHÂN LOẠI CÁC FEATURES
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
categorical_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns 
                    if col not in binary_cols]
numerical_cols = [col for col in X_train.select_dtypes(include=['int64', 'float64']).columns 
                  if col not in binary_cols]

# 3. XÂY DỰNG PIPELINE TIỀN XỬ LÝ (KHÔNG IMPUTE, KHÔNG SCALE)
# - Binary: Map về số, NaNs tự động thành -1
binary_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)

# - Categorical: One-Hot, bỏ qua giá trị lạ và NaN
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# - Numerical: Không biến đổi, để mô hình tự xử lý NaN
numerical_transformer = 'passthrough'

preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_transformer, binary_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

# 4. CHỌN MÔ HÌNH VÀ HUẤN LUYỆN
# HistGradientBoosting tự động xử lý missing values, hỗ trợ class_weight
model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

pipeline.fit(X_train, y_train)

# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING ---")
print("\n1. Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n2. Classification Report:\n", classification_report(y_test, y_pred))
print(f"3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")

--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING ---

1. Confusion Matrix:
 [[15997  4405]
 [ 1175  4455]]

2. Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.78      0.85     20402
           1       0.50      0.79      0.61      5630

    accuracy                           0.79     26032
   macro avg       0.72      0.79      0.73     26032
weighted avg       0.84      0.79      0.80     26032

3. ROC-AUC Score: 0.8639
4. PR-AUC (Average Precision): 0.6321


In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

# ==========================================
# 1. CLASS CUSTOM FEATURE ENGINEERING
# ==========================================
class AdvancedFeatureEngineer(BaseEstimator, TransformerMixin):
    def _create_buckets_and_profiles(self, df):
        df['age_bucket'] = pd.cut(df['age'], bins=[0, 30, 50, 70, 150], labels=['young', 'adult', 'senior', 'elderly']).astype(str)
        df['bmi_bucket'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=['underweight', 'normal', 'overweight', 'obese']).astype(str)
        
        if 'd1_sysbp_noninvasive_max' in df.columns:
            df['bp_bucket'] = pd.cut(df['d1_sysbp_noninvasive_max'], bins=[0, 120, 130, 140, 300], labels=['normal', 'elevated', 'high_1', 'high_2']).astype(str)
        else:
            df['bp_bucket'] = 'unknown'
            
        if 'd1_glucose_max' in df.columns:
            df['glucose_bucket'] = pd.cut(df['d1_glucose_max'], bins=[0, 140, 200, 1000], labels=['normal', 'prediabetes', 'diabetes']).astype(str)
        else:
            df['glucose_bucket'] = 'unknown'
            
        df['profile'] = df['age_bucket'] + "_" + df['bmi_bucket'] + "_" + df['ethnicity'].astype(str) + "_" + df['gender'].astype(str)
        return df

    def fit(self, X, y=None):
        self.freq_encoding_maps_ = {}
        self.agg_maps_ = {}
        
        # 1. Frequency Encoding Mapping
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            self.freq_encoding_maps_[col] = X[col].value_counts(normalize=True).to_dict()
            
        # 2. MICE Imputer theo gender
        self.imputers_ = {}
        for gender in X['gender'].dropna().unique():
            imputer = IterativeImputer(random_state=42)
            mask = X['gender'] == gender
            imputer.fit(X.loc[mask, ['age', 'height', 'weight']])
            self.imputers_[gender] = imputer
            
        # 3. Học Mean Aggregate từ tập Train
        df_fit = X.copy()
        for gender, imputer in self.imputers_.items():
            mask = df_fit['gender'] == gender
            if mask.sum() > 0:
                df_fit.loc[mask, ['age', 'height', 'weight']] = imputer.transform(df_fit.loc[mask, ['age', 'height', 'weight']])
        df_fit['bmi'] = df_fit['weight'] / ((df_fit['height'] / 100) ** 2)
        
        for min_col in [c for c in df_fit.columns if c.endswith('_min')]:
            max_col = min_col.replace('_min', '_max')
            base_name = min_col.replace('_min', '')
            if max_col in df_fit.columns:
                df_fit[f'{base_name}_avg'] = (df_fit[max_col] + df_fit[min_col]) / 2
                
        df_fit = self._create_buckets_and_profiles(df_fit)
        
        avg_cols = [c for c in df_fit.columns if c.endswith('_avg')][:10] 
        group_cols = ['profile', 'apache_3j_diagnosis', 'bp_bucket', 'glucose_bucket']
        
        for group in group_cols:
            if group in df_fit.columns:
                self.agg_maps_[group] = df_fit.groupby(group)[avg_cols].mean()
                
        return self

    def transform(self, X):
        df = X.copy()
        
        # 1. DATA CLEANING
        # Flip min/max an toàn
        min_cols = [c for c in df.columns if c.endswith('_min')]
        for min_col in min_cols:
            max_col = min_col.replace('_min', '_max')
            if max_col in df.columns:
                mask = df[min_col] > df[max_col]
                if mask.any():
                    temp = df.loc[mask, min_col].copy()
                    df.loc[mask, min_col] = df.loc[mask, max_col]
                    df.loc[mask, max_col] = temp

        # MICE Imputation by Gender
        for gender, imputer in self.imputers_.items():
            mask = df['gender'] == gender
            if mask.sum() > 0:
                df.loc[mask, ['age', 'height', 'weight']] = imputer.transform(df.loc[mask, ['age', 'height', 'weight']])
        
        # Recalculate BMI
        df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

        # Drop redundant BP features
        bp_drop = [c for c in df.columns if re.search(r'(?<!invasive)_?(mbp|sysbp|diasbp)_(max|min)$', c)]
        df = df.drop(columns=bp_drop, errors='ignore')

        # 2. FEATURE ENGINEERING
        d1_h1_cols = [c for c in df.columns if c.startswith('d1_') or c.startswith('h1_')]
        for col in d1_h1_cols:
            df[f'{col}_is_missing'] = df[col].isnull().astype(int)

        # Basic Math
        for min_col in [c for c in df.columns if c.endswith('_min')]:
            max_col = min_col.replace('_min', '_max')
            base_name = min_col.replace('_min', '')
            if max_col in df.columns:
                df[f'{base_name}_diff'] = df[max_col] - df[min_col]
                df[f'{base_name}_ratio'] = df[max_col] / (df[min_col] + 1e-5)
                df[f'{base_name}_avg'] = (df[max_col] + df[min_col]) / 2

        # Pulse Pressure
        for prefix in ['d1', 'h1']:
            for suffix in ['max', 'min', 'noninvasive_max', 'noninvasive_min', 'invasive_max', 'invasive_min']:
                sys_col = f'{prefix}_sysbp_{suffix}'
                dia_col = f'{prefix}_diasbp_{suffix}'
                if sys_col in df.columns and dia_col in df.columns:
                    df[f'{prefix}_pulse_pressure_{suffix}'] = df[sys_col] - df[dia_col]

        # d1 vs h1 comparisons
        h1_cols = [c for c in df.columns if c.startswith('h1_')]
        for h1_col in h1_cols:
            d1_col = h1_col.replace('h1_', 'd1_', 1)
            if d1_col in df.columns and df[d1_col].dtype in ['float64', 'int64']:
                df[f'd1_h1_diff_{h1_col[3:]}'] = df[d1_col] - df[h1_col]
                df[f'd1_h1_ratio_{h1_col[3:]}'] = df[d1_col] / (df[h1_col] + 1e-5)

        # 2nd level combos
        h1_diff_cols = [c for c in df.columns if c.startswith('h1_') and c.endswith('_diff')]
        for h1_diff in h1_diff_cols:
            d1_diff = h1_diff.replace('h1_', 'd1_', 1)
            if d1_diff in df.columns:
                df[f'diff_of_diffs_{h1_diff[3:-5]}'] = df[d1_diff] - df[h1_diff]

        # Buckets & Profiles
        df = self._create_buckets_and_profiles(df)

        # Frequency Encoding
        for col, mapping in self.freq_encoding_maps_.items():
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(mapping).fillna(0)

        # Aggregated features by groups
        avg_cols = [c for c in df.columns if c.endswith('_avg')][:10]
        group_cols = ['profile', 'apache_3j_diagnosis', 'bp_bucket', 'glucose_bucket']
        for group in group_cols:
            if group in df.columns and group in self.agg_maps_:
                mean_df = self.agg_maps_[group]
                for col in avg_cols:
                    if col in mean_df.columns:
                        df[f'{col}_mean_by_{group}'] = df[group].map(mean_df[col])

        return df

# ==========================================
# 2. FEATURE SELECTION FUNCTION (TỐI ƯU RAM)
# ==========================================
def select_features_hierarchical(X_train, y_train, model):
    print("Bắt đầu Feature Selection...")
    
    # 1. Downcast dữ liệu
    X_train_float32 = X_train.astype(np.float32)
    
    # 2. Subsample cho Correlation (~15,000 dòng)
    print("- Đang tính ma trận tương quan...")
    n_samples_corr = min(15000, X_train_float32.shape[0])
    X_sample_corr = X_train_float32.sample(n=n_samples_corr, random_state=42)
    
    corr = spearmanr(X_sample_corr.fillna(0)).correlation
    corr = np.nan_to_num(corr, nan=0.0) 
    corr = (corr + corr.T) / 2
    np.fill_diagonal(corr, 1)
    
    distance_matrix = 1 - np.abs(corr)
    np.fill_diagonal(distance_matrix, 0) 
    
    condensed_dist = squareform(distance_matrix) 
    dist_linkage = hierarchy.ward(condensed_dist)
    cluster_ids = hierarchy.fcluster(dist_linkage, t=0.5, criterion='distance')
    
    # 3. Subsample cho Permutation Importance
    print("- Đang huấn luyện và tính Permutation Importance...")
    model.fit(X_train_float32, y_train)
    
    X_pi_sample, _, y_pi_sample, _ = train_test_split(
        X_train_float32, y_train, train_size=0.15, stratify=y_train, random_state=42
    )
    
    # n_jobs=2 để tránh cấp số nhân RAM
    result = permutation_importance(model, X_pi_sample, y_pi_sample, n_repeats=3, random_state=42, n_jobs=2)
    importances = result.importances_mean
    
    selected_features = []
    for cluster_id in np.unique(cluster_ids):
        cluster_indices = np.where(cluster_ids == cluster_id)[0]
        cluster_importances = importances[cluster_indices]
        
        best_idx = cluster_indices[np.argmax(cluster_importances)]
        
        if importances[best_idx] > 0.001:
            selected_features.append(X_train.columns[best_idx])
            
    print(f"Đã chọn {len(selected_features)} / {X_train.shape[1]} features.")
    return selected_features

# ==========================================
# 3. LUỒNG CHÍNH (MAIN PIPELINE)
# ==========================================
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

# XÓA DÒNG: Tuổi = 0 hoặc NaN
data = data[(data['age'] > 0) & (data['age'].notnull())].copy()

# XÓA CỘT RÁC ĐỂ TRÁNH TRÀN ONE-HOT ENCODING
drop_cols = [target_col, 'encounter_id', 'hospital_id']
X = data.drop(columns=[c for c in drop_cols if c in data.columns])
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Khởi tạo Custom Engineer
engineer = AdvancedFeatureEngineer()
X_train_eng = engineer.fit_transform(X_train)
X_test_eng = engineer.transform(X_test)

# Xác định lại loại cột sau Feature Engineering
binary_cols = [col for col in X_train_eng.columns if X_train_eng[col].nunique() == 2]
categorical_cols = [col for col in X_train_eng.select_dtypes(include=['object', 'category']).columns if col not in binary_cols]
numerical_cols = [col for col in X_train_eng.select_dtypes(include=['number']).columns if col not in binary_cols]

# Cập nhật Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('bin', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), binary_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [c for c in categorical_cols if c not in ['icu_id', 'apache_2_diagnosis']]),
    ('target_enc', TargetEncoder(target_type='binary'), [c for c in ['icu_id', 'apache_2_diagnosis'] if c in X_train_eng.columns]),
    ('num', 'passthrough', numerical_cols)
])

# Feature Selection
X_train_processed = pd.DataFrame(preprocessor.fit_transform(X_train_eng, y_train), columns=preprocessor.get_feature_names_out())
X_test_processed = pd.DataFrame(preprocessor.transform(X_test_eng), columns=preprocessor.get_feature_names_out())

model_fs = HistGradientBoostingClassifier(random_state=42)
final_features = select_features_hierarchical(X_train_processed, y_train, model_fs)

# Filter lại tập dữ liệu
X_train_final = X_train_processed[final_features]
X_test_final = X_test_processed[final_features]

# Huấn luyện mô hình cuối cùng
final_model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
final_model.fit(X_train_final, y_train)

# Đánh giá
y_pred = final_model.predict(X_test_final)
y_pred_proba = final_model.predict_proba(X_test_final)[:, 1]

print("\n--- ĐÁNH GIÁ MÔ HÌNH ---")
print("1. Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("2. Classification Report:\n", classification_report(y_test, y_pred))
print(f"3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC Score: {average_precision_score(y_test, y_pred_proba):.4f}")

D:\ml_cache\temp\ipykernel_25224\945962025.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_is_missing'] = df[col].isnull().astype(int)
D:\ml_cache\temp\ipykernel_25224\945962025.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_is_missing'] = df[col].isnull().astype(int)
D:\ml_cache\temp\ipykernel_25224\945962025.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at 

Bắt đầu Feature Selection...
- Đang tính ma trận tương quan...


c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


- Đang huấn luyện và tính Permutation Importance...
Đã chọn 11 / 1240 features.

--- ĐÁNH GIÁ MÔ HÌNH ---
1. Confusion Matrix:
 [[14829  4701]
 [ 1150  4348]]
2. Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.76      0.84     19530
           1       0.48      0.79      0.60      5498

    accuracy                           0.77     25028
   macro avg       0.70      0.78      0.72     25028
weighted avg       0.83      0.77      0.78     25028

3. ROC-AUC Score: 0.8543
4. PR-AUC Score: 0.6166
